# DoubleB Particle Transformer — H→bb vs QCD
Particle Transformer notebook with CMS-style physics labels.

In [ ]:
import os
import glob
import numpy as np
import awkward as ak
import uproot
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import roc_curve, auc
import matplotlib.pyplot as plt

In [ ]:
# === Paths ===
train_directory = "~/Desktop/DoubleB/train_data/"
test_directory = "~/Desktop/DoubleB/test_data/"

# === Feature lists (edit if needed) ===
jet_features = []  # add jet features here
pf_features = []   # add PF features here
spectators = ['fj_sdmass', 'fj_pt']

In [ ]:
def load_data(directory, chunk_size=2000, max_files=None):
    files = glob.glob(os.path.expanduser(directory) + "*.root")
    if max_files:
        files = files[:max_files]

    jet_arrays, pf_arrays, label_arrays = [], [], []

    for f in files:
        print("Reading:", f)
        try:
            tree = uproot.open(f)["deepntuplizer/tree"]
        except Exception as e:
            print("Skipping corrupted file:", f, e)
            continue

        for batch in tree.iterate(
            jet_features + pf_features + spectators +
            ['sample_isQCD', 'fj_isQCD', 'fj_isH', 'fj_isBB'],
            step_size=chunk_size,
            library='ak'
        ):

            mask = (
                (batch['fj_sdmass'] > 40)
                & (batch['fj_sdmass'] < 200)
                & (batch['fj_pt'] > 350)
                & (batch['fj_pt'] < 2000)
            )
            batch = batch[mask]

            label_qcd = batch['fj_isQCD'] * batch['sample_isQCD']
            label_hbb = batch['fj_isH'] * batch['fj_isBB']

            valid = (label_qcd == 1) | (label_hbb == 1)
            batch = batch[valid]

            y = ak.where(label_hbb[valid] == 1, 1, 0)

            jet_arrays.append(batch[jet_features])
            pf_arrays.append(batch[pf_features])
            label_arrays.append(y)

    jets = ak.concatenate(jet_arrays)
    pfs = ak.concatenate(pf_arrays)
    labels = ak.concatenate(label_arrays)

    print("Loaded jets:", len(jets))
    return jets, pfs, labels

In [ ]:
class ParticleDataset(Dataset):
    def __init__(self, jets, pfs, labels):
        self.jets = jets
        self.pfs = pfs
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        jet = torch.tensor(ak.to_numpy(self.jets[idx]), dtype=torch.float32)
        pf = torch.tensor(ak.to_numpy(self.pfs[idx]), dtype=torch.float32)
        y = torch.tensor(self.labels[idx], dtype=torch.float32)
        return pf, jet, y

In [ ]:
class ParticleTransformer(nn.Module):
    def __init__(self, pf_dim, jet_dim, hidden=128):
        super().__init__()
        self.pf_embed = nn.Linear(pf_dim, hidden)
        encoder = nn.TransformerEncoderLayer(hidden, nhead=4)
        self.transformer = nn.TransformerEncoder(encoder, num_layers=2)
        self.jet_fc = nn.Linear(jet_dim, hidden)
        self.out = nn.Linear(hidden * 2, 1)

    def forward(self, pfs, jets):
        x = self.pf_embed(pfs)
        x = self.transformer(x)
        x = x.mean(dim=1)
        j = self.jet_fc(jets)
        z = torch.cat([x, j], dim=-1)
        return torch.sigmoid(self.out(z)).squeeze()

In [ ]:
def train_epoch(model, loader, optimizer, criterion):
    model.train()
    total = 0
    for pfs, jets, y in loader:
        optimizer.zero_grad()
        preds = model(pfs, jets)
        loss = criterion(preds, y)
        loss.backward()
        optimizer.step()
        total += loss.item()
    return total / len(loader)

In [ ]:
def evaluate(model, loader):
    model.eval()
    scores, labels = [], []
    with torch.no_grad():
        for pfs, jets, y in loader:
            s = model(pfs, jets)
            scores.extend(s.numpy())
            labels.extend(y.numpy())

    scores = np.array(scores)
    labels = np.array(labels)

    fpr, tpr, _ = roc_curve(labels, scores)
    roc_auc = auc(fpr, tpr)

    plt.figure(figsize=(6,6))
    plt.plot(tpr, 1-fpr, label=f"AUC={roc_auc:.3f}")
    plt.xlabel("Signal efficiency")
    plt.ylabel("Background rejection")
    plt.legend()
    plt.grid(True)
    plt.show()